In [ ]:
import pandas as pd
from langdetect import detect, LangDetectException
from tqdm import tqdm

df_raw = pd.read_csv('../data/raw_pushshift.csv')  # Your full 3.5M+ raw entries

# Verify dialect column exists and count current sizes
print("Current dialect distribution:")
print(df_raw['dialect'].value_counts().sort_index())

# Target: 10k TOTAL proportional samples
total_target = 10000

# Proportional targets based on your earlier counts
# rio: 982k / 2.2M ≈ 45%, cdmx: 44%, madrid: 11%
rio_prop = 982124 / 2198705
cdmx_prop = 971345 / 2198705  
madrid_prop = 245236 / 2198705

rio_target = int(total_target * rio_prop)    # ~4490
cdmx_target = int(total_target * cdmx_prop)  # ~4420
madrid_target = total_target - rio_target - cdmx_target  # ~1090

print(f"\nSampling targets:")
print(f"rioplatense: {rio_target:,}")
print(f"cdmx:        {cdmx_target:,}") 
print(f"madrid:      {madrid_target:,}")
print(f"Total:       {rio_target + cdmx_target + madrid_target:,}")

# Stratified sampling by dialect (preserves proportions)
# FIXED: Sample each dialect separately, then concat
raw_samples_rio = df_raw[df_raw['dialect'] == 'rioplatense'].sample(n=rio_target, random_state=42)
raw_samples_cdmx = df_raw[df_raw['dialect'] == 'cdmx'].sample(n=cdmx_target, random_state=42)
raw_samples_madrid = df_raw[df_raw['dialect'] == 'madrid'].sample(n=madrid_target, random_state=42)

raw_samples = pd.concat([raw_samples_rio, raw_samples_cdmx, raw_samples_madrid], ignore_index=True)

print(f"\n✅ Sampled {len(raw_samples):,} RAW entries")
print("\nFinal sample distribution:")
print(raw_samples['dialect'].value_counts().sort_index())

Current dialect distribution:
dialect
cdmx            9715219
madrid           512353
rioplatense    18968338
Name: count, dtype: int64

Sampling targets:
rioplatense: 4,466
cdmx:        4,417
madrid:      1,117
Total:       10,000

✅ Sampled 10,000 RAW entries

Final sample distribution:
dialect
cdmx           4417
madrid         1117
rioplatense    4466
Name: count, dtype: int64


In [ ]:
# Light RAW filter (remove obvious garbage ONLY)
def light_raw_filter(df):
    """Minimal filtering - keep ALL sentiment signals"""
    return (df
        .query("text.str.len() >= 20")      # No micro-comments
        .drop_duplicates(subset=['text'])    # No copy-pastes
        # NO OTHER CHANGES - preserve emojis, URLs, casing, repetitions
    )
  
tqdm.pandas(desc="Detecting language")

def is_spanish(text):
    try:
        return detect(text) == "es"
    except LangDetectException:
        return False

# Apply light filter
raw_filtered = light_raw_filter(raw_samples)

print("Running language detection, this will take a while...")
raw_filtered["is_spanish"] = raw_filtered["text"].progress_apply(is_spanish)
raw_filtered = raw_filtered[raw_filtered["is_spanish"]].drop(columns="is_spanish")
print(f"After language filter:\n{raw_filtered.groupby('dialect').size()}")

print(f"Removed: {len(raw_samples) - len(raw_filtered):,} entries")
print(f"Remaining: {len(raw_filtered):,} RAW entries for labeling")

# Save results
raw_filtered.to_csv('raw_10k_for_labeling.csv', index=False)

# Save hashes for later exclusion from full training set
raw_filtered['text_hash'] = raw_filtered['text'].apply(hash)
raw_filtered[['text', 'text_hash', 'dialect']].to_csv('labeling_hashes.csv', index=False)

Running language detection, this will take a while...


Detecting language: 100%|██████████| 8899/8899 [00:33<00:00, 267.16it/s]


After language filter:
dialect
cdmx           3540
madrid          698
rioplatense    3554
dtype: int64

After light filter:
Removed: 2,208 entries
Remaining: 7,792 RAW entries for labeling
dialect
cdmx           3540
madrid          698
rioplatense    3554
Name: count, dtype: int64

✅ FILES CREATED:
  raw_10k_for_labeling.csv     ← LABEL THIS (8-9k entries)
  labeling_hashes.csv          ← EXCLUDE FROM TRAINING LATER

Next: Manually label ~1.5k per dialect from raw_10k_for_labeling.csv
